# Superstore Sales & Profitability Analysis — 02. Data Cleaning

## 1. Objective

Prepare **minimal, auditable** cleaned copies of the Superstore sheets and write them to
`data/processed/`, without changing the raw workbook and without deleting or correcting any row.

This notebook:

- loads the three raw sheets (`Orders`, `People`, `Returns`),
- applies only the small set of approved transformations,
- documents (but does **not** remove) repeated keys and duplicated records,
- validates the processed outputs against the source,
- records every output file and every transformation in a change log.

This notebook does **not** perform any sales or profitability analysis and makes no
business-performance claims.

## 2. Cleaning Rules (Approved Decisions)

The following decisions are applied exactly:

1. **Preserve raw source files unchanged.** Raw data is read-only; outputs go only to `data/processed/`.
2. **Never automatically delete or correct rows.** No row is dropped or edited.
3. **Keep repeated `Order ID` values.** An order legitimately contains multiple product lines.
4. **Inspect, do not remove, repeated `Order ID + Product ID` cases and rows duplicated when ignoring `Row ID`.** These are documented only.
5. **Keep future-dated orders.** They are not assumed to be invalid.
6. **Handle `Postal Code` carefully.** Do not guess missing leading zeros. Preserve original values in the raw data and document any representation change.
7. **Do not force a one-to-one mapping between `Product ID` and `Product Name`.**
8. **Keep `Returns` at order level.** Avoid joins that multiply order-line records.
9. **Validate that `People` covers all order regions.**

Where a transformation is not necessary, the data is left unchanged and the reason is stated.

## 3. Environment & Imports

In [1]:
from pathlib import Path
import hashlib
import sys

import numpy as np
import pandas as pd


def repository_relative(path) -> str:
    '''Return *path* relative to the repository root when it is inside it.'''
    absolute = Path(path).absolute()
    for base in (Path.cwd(), *Path.cwd().parents):
        if (base / "data" / "raw").is_dir():
            try:
                return str(absolute.relative_to(base))
            except ValueError:
                break
    return absolute.name


print("Python executable:", repository_relative(sys.executable))
print("Python version:   ", sys.version.split()[0])
print("pandas version:   ", pd.__version__)
print("numpy version:    ", np.__version__)

Python executable: .venv/bin/python
Python version:    3.14.8
pandas version:    3.0.6
numpy version:     2.5.4


## 4. Data Loading

In [2]:
def find_repository_root(start_directory: Path) -> Path:
    '''Locate the repository root that contains data/raw/sample_-_superstore.xls.'''
    for candidate in [start_directory, *start_directory.parents]:
        if (candidate / "data" / "raw" / "sample_-_superstore.xls").exists():
            return candidate
    raise FileNotFoundError(
        "Could not locate the repository root containing data/raw/sample_-_superstore.xls."
    )


def file_sha256(path: Path) -> str:
    '''Return the SHA-256 checksum of a file.'''
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(65536), b""):
            digest.update(chunk)
    return digest.hexdigest()


repository_root = find_repository_root(Path.cwd())
raw_workbook_path = repository_root / "data" / "raw" / "sample_-_superstore.xls"
processed_directory = repository_root / "data" / "processed"

print("Notebook working directory:", repository_relative(Path.cwd()))
print("Repository root:          ", repository_root.name)
print("Raw workbook:             ", repository_relative(raw_workbook_path))
print("Processed directory:      ", repository_relative(processed_directory))

raw_sha256_before = file_sha256(raw_workbook_path)
print("Raw workbook SHA-256 (before):", raw_sha256_before)

Notebook working directory: notebooks
Repository root:           salesAnalysis
Raw workbook:              data/raw/sample_-_superstore.xls
Processed directory:       data/processed
Raw workbook SHA-256 (before): 5045ed4fec49cae2cc0dfc5ab0d204ba09efa767fe00c996822499e8cc42fbbd


In [3]:
raw_orders = pd.read_excel(raw_workbook_path, sheet_name="Orders", engine="xlrd")
raw_people = pd.read_excel(raw_workbook_path, sheet_name="People", engine="xlrd")
raw_returns = pd.read_excel(raw_workbook_path, sheet_name="Returns", engine="xlrd")

raw_sheets = {"Orders": raw_orders, "People": raw_people, "Returns": raw_returns}

print("Loaded raw sheets:")
for sheet_name, sheet_data in raw_sheets.items():
    print(f"  {sheet_name}: {sheet_data.shape[0]} rows x {sheet_data.shape[1]} columns")

Loaded raw sheets:
  Orders: 10194 rows x 21 columns
  People: 4 rows x 2 columns
  Returns: 296 rows x 2 columns


## 5. Transformations

Only the following minimal transformations are applied. No row is added, removed, or corrected.

- **Dates** (`Order Date`, `Ship Date`): explicitly parsed as datetime. Values are unchanged.
- **`Postal Code`**: stored consistently as text **without padding**. Numeric values such as
  `1040` become `"1040"` (no leading zero is added or invented); text values such as `"05401"`
  are kept exactly as stored. This is a representation change only, documented below.
- **`Quantity`**: already an integer type; no change needed.
- **All other columns / rows**: left unchanged.

Rows and columns are preserved in full (`Orders` keeps all 21 columns and all rows).

In [4]:
orders_clean = raw_orders.copy()
people_clean = raw_people.copy()
returns_clean = raw_returns.copy()

# Dates: parse explicitly (values are unchanged).
orders_clean["Order Date"] = pd.to_datetime(orders_clean["Order Date"])
orders_clean["Ship Date"] = pd.to_datetime(orders_clean["Ship Date"])


def postal_code_to_text(value):
    '''Represent a stored Postal Code value as text without adding leading zeros.'''
    if isinstance(value, str):
        return value
    if isinstance(value, float) and value.is_integer():
        return str(int(value))
    if isinstance(value, (int, np.integer)):
        return str(int(value))
    return str(value)


orders_clean["Postal Code"] = (
    orders_clean["Postal Code"].map(postal_code_to_text).astype("string")
)

print("Transformed copies created:")
for name, frame in {"orders_clean": orders_clean, "people_clean": people_clean, "returns_clean": returns_clean}.items():
    print(f"  {name}: {frame.shape[0]} rows x {frame.shape[1]} columns")

Transformed copies created:
  orders_clean: 10194 rows x 21 columns
  people_clean: 4 rows x 2 columns
  returns_clean: 296 rows x 2 columns


## 6. Duplicate Documentation (No Rows Removed)

Repeated and duplicated records are documented here and kept in full. `Order ID` alone is not a
key (orders contain multiple lines), so duplicates are examined at three levels:

- exact duplicate rows (all columns),
- repeated `Order ID + Product ID` combinations,
- rows duplicated when ignoring the unique `Row ID` column.

In [5]:
print("Exact duplicate rows in raw Orders:", int(raw_orders.duplicated().sum()))
print()

repeated_order_product_mask = orders_clean.duplicated(
    subset=["Order ID", "Product ID"], keep=False
)
repeated_order_product = (
    orders_clean.loc[repeated_order_product_mask]
    .sort_values(["Order ID", "Product ID"])
)
print(
    "Rows in repeated Order ID + Product ID groups:",
    int(repeated_order_product_mask.sum()),
    f"({repeated_order_product[['Order ID', 'Product ID']].drop_duplicates().shape[0]} distinct combinations)",
)
display(
    repeated_order_product[
        ["Row ID", "Order ID", "Product ID", "Product Name", "Sales", "Quantity", "Discount", "Profit"]
    ]
)

business_columns = [column for column in raw_orders.columns if column != "Row ID"]
duplicate_record_mask = raw_orders.duplicated(subset=business_columns, keep=False)
print(
    "Rows duplicated when ignoring Row ID:",
    int(duplicate_record_mask.sum()),
    "(2 identical record pairs; kept, not removed)",
)
display(
    raw_orders.loc[
        duplicate_record_mask,
        ["Row ID", "Order ID", "Product ID", "Order Date", "Sales", "Quantity", "Discount", "Profit"],
    ].sort_values(["Order ID", "Product ID"])
)

Exact duplicate rows in raw Orders: 0

Rows in repeated Order ID + Product ID groups: 22 (11 distinct combinations)


,Row ID,Order ID,Product ID,Product Name,Sales,Quantity,Discount,Profit
2017,2018,CA-2023-123625,FUR-FU-10004093,Hand-Finished Solid Wood Document Frame,101.120,8,0.0,37.4144
2019,2020,CA-2023-123625,FUR-FU-10004093,Hand-Finished Solid Wood Document Frame,68.460,2,0.0,20.5380
2020,2021,CA-2023-123625,OFF-FA-10000089,Acco Glide Clips,68.460,2,0.0,20.5380
2022,2023,CA-2023-123625,OFF-FA-10000089,Acco Glide Clips,19.600,5,0.0,9.6040
1698,1699,CA-2023-153623,FUR-FU-10002501,Nu-Dell Executive Frame,99.120,8,0.0,35.4144
1699,1700,CA-2023-153623,FUR-FU-10002501,Nu-Dell Executive Frame,99.120,8,0.0,35.4144
390,391,US-2023-150119,FUR-CH-10002965,Global Leather Highback Executive Chair with P...,281.372,2,0.3,-12.0588
391,392,US-2023-150119,FUR-CH-10002965,Global Leather Highback Executive Chair with P...,281.372,2,0.3,-12.0588
2879,2880,US-2024-103135,OFF-BI-10000069,"GBC Prepunched Paper, 19-Hole, for Binding Sys...",135.090,9,0.0,62.1414
2880,2881,US-2024-103135,OFF-BI-10000069,"GBC Prepunched Paper, 19-Hole, for Binding Sys...",90.060,6,0.0,41.4276


Rows duplicated when ignoring Row ID: 4 (2 identical record pairs; kept, not removed)


,Row ID,Order ID,Product ID,Order Date,Sales,Quantity,Discount,Profit
1698,1699,CA-2023-153623,FUR-FU-10002501,2023-11-24,99.120,8,0.0,35.4144
1699,1700,CA-2023-153623,FUR-FU-10002501,2023-11-24,99.120,8,0.0,35.4144
390,391,US-2023-150119,FUR-CH-10002965,2023-04-23,281.372,2,0.3,-12.0588
391,392,US-2023-150119,FUR-CH-10002965,2023-04-23,281.372,2,0.3,-12.0588


## 7. Postal Code Representation

`Postal Code` is intentionally **not** forced into a numeric type, and no missing leading zero is
guessed. The raw column mixes Python `int` and `str` values; in the processed copy it is stored
consistently as text while preserving each stored value exactly. The distribution of value types
and string lengths is reported before and after the representation change.

No value is padded, truncated, or otherwise altered.

In [6]:
before_types = raw_orders["Postal Code"].map(lambda value: type(value).__name__).value_counts()
after_types = orders_clean["Postal Code"].map(lambda value: type(value).__name__).value_counts()
print("Postal Code value types BEFORE:", before_types.to_dict())
print("Postal Code value types AFTER: ", after_types.to_dict())
print()

before_lengths = raw_orders["Postal Code"].astype(str).str.len()
after_lengths = orders_clean["Postal Code"].str.len()
print("Postal Code string lengths BEFORE:", before_lengths.value_counts().sort_index().to_dict())
print("Postal Code string lengths AFTER: ", after_lengths.value_counts().sort_index().to_dict())
print()

length_by_country = pd.crosstab(orders_clean["Country/Region"], after_lengths)
print("Postal Code string length by Country/Region (processed):")
display(length_by_country)
print()

values_preserved = (
    raw_orders["Postal Code"].map(postal_code_to_text).tolist()
    == orders_clean["Postal Code"].tolist()
)
print("Every Postal Code value preserved without padding:", bool(values_preserved))
print("Four-character United States Postal Codes:", int(length_by_country.loc["United States", 4]) if 4 in length_by_country.columns else 0)

Postal Code value types BEFORE: {'int': 9983, 'str': 211}
Postal Code value types AFTER:  {'str': 10194}

Postal Code string lengths BEFORE: {3: 200, 4: 438, 5: 9556}
Postal Code string lengths AFTER:  {np.int64(3): 200, np.int64(4): 438, np.int64(5): 9556}

Postal Code string length by Country/Region (processed):


Postal Code,3,4,5
Country/Region,,,
Canada,200,0,0
United States,0,438,9556



Every Postal Code value preserved without padding: True
Four-character United States Postal Codes: 438


## 8. Validation Against Source

In [7]:
validation_results = []


def record_check(check_name, passed, detail):
    validation_results.append({"check": check_name, "passed": bool(passed), "detail": detail})


record_check("Orders row count preserved", len(orders_clean) == len(raw_orders), f"{len(orders_clean)} == {len(raw_orders)}")
record_check("Orders column count preserved", orders_clean.shape[1] == raw_orders.shape[1], f"{orders_clean.shape[1]} == {raw_orders.shape[1]}")
record_check("People row count preserved", len(people_clean) == len(raw_people), f"{len(people_clean)} == {len(raw_people)}")
record_check("Returns row count preserved", len(returns_clean) == len(raw_returns), f"{len(returns_clean)} == {len(raw_returns)}")

record_check("Row ID remains unique", bool(orders_clean["Row ID"].is_unique), f"{orders_clean['Row ID'].nunique()} unique values")

source_lines_per_order = raw_orders.groupby("Order ID").size().sort_index()
processed_lines_per_order = orders_clean.groupby("Order ID").size().sort_index()
record_check(
    "Order ID line counts unchanged",
    source_lines_per_order.equals(processed_lines_per_order),
    f"{orders_clean['Order ID'].nunique()} distinct Order IDs",
)

record_check("Order Date is datetime", pd.api.types.is_datetime64_any_dtype(orders_clean["Order Date"]), str(orders_clean["Order Date"].dtype))
record_check("Ship Date is datetime", pd.api.types.is_datetime64_any_dtype(orders_clean["Ship Date"]), str(orders_clean["Ship Date"].dtype))
record_check("Quantity is integer", pd.api.types.is_integer_dtype(orders_clean["Quantity"]), str(orders_clean["Quantity"].dtype))
record_check(
    "Sales / Discount / Profit are numeric",
    all(pd.api.types.is_numeric_dtype(orders_clean[column]) for column in ["Sales", "Discount", "Profit"]),
    "float64",
)

record_check("Postal Code values preserved (no padding)", bool(values_preserved), "element-wise equality with raw values")
record_check("No exact duplicate rows introduced", int(orders_clean.duplicated().sum()) == 0, f"{int(orders_clean.duplicated().sum())} duplicates")

order_ids = set(orders_clean["Order ID"].unique())
return_order_ids = set(returns_clean["Order ID"].unique())
record_check("Returns Order ID is unique", bool(returns_clean["Order ID"].is_unique), f"{returns_clean['Order ID'].nunique()} distinct")
record_check("Every return matches an order", return_order_ids <= order_ids, f"{len(return_order_ids - order_ids)} unmatched")

orders_regions = set(orders_clean["Region"].dropna().unique())
people_regions = set(people_clean["Region"].dropna().unique())
record_check("People covers all order regions", orders_regions <= people_regions, f"missing: {sorted(orders_regions - people_regions)}")

validation_table = pd.DataFrame(validation_results)
display(validation_table)

assert validation_table["passed"].all(), "One or more validation checks failed."
print("All validation checks passed.")

,check,passed,detail
0,Orders row count preserved,True,10194 == 10194
1,Orders column count preserved,True,21 == 21
2,People row count preserved,True,4 == 4
3,Returns row count preserved,True,296 == 296
4,Row ID remains unique,True,10194 unique values
5,Order ID line counts unchanged,True,5111 distinct Order IDs
6,Order Date is datetime,True,datetime64[us]
7,Ship Date is datetime,True,datetime64[us]
8,Quantity is integer,True,int64
9,Sales / Discount / Profit are numeric,True,float64


All validation checks passed.


## 9. Save Processed Outputs

Processed files are written **only** under `data/processed/`. The raw workbook is never written to.
Saved files are read back to confirm their row counts.

In [8]:
processed_directory.mkdir(parents=True, exist_ok=True)

output_frames = {
    "orders_clean.csv": orders_clean,
    "people_clean.csv": people_clean,
    "returns_clean.csv": returns_clean,
}

created_files = []
for file_name, frame in output_frames.items():
    output_path = processed_directory / file_name
    frame.to_csv(output_path, index=False)
    created_files.append(output_path)
    reread = pd.read_csv(output_path)
    print(f"Wrote {output_path.name}: {reread.shape[0]} rows x {reread.shape[1]} columns")

print()
print("Created files:")
for output_path in created_files:
    print(" ", output_path.relative_to(repository_root))

Wrote orders_clean.csv: 10194 rows x 21 columns
Wrote people_clean.csv: 4 rows x 2 columns
Wrote returns_clean.csv: 296 rows x 2 columns

Created files:
  data/processed/orders_clean.csv
  data/processed/people_clean.csv
  data/processed/returns_clean.csv


## 10. Change Log

In [9]:
# Fixed project analysis date (2026-10-13); do not derive it from wall-clock time.
analysis_date = pd.Timestamp("2026-10-13")
future_order_mask = pd.to_datetime(raw_orders["Order Date"]) > analysis_date
file_log_rows = [
    {"step": 1, "target": "orders_clean.csv", "action": "Copied all rows and columns unchanged", "rationale": "Preserve every record; no row deletion or correction", "rows_affected": 0},
    {"step": 2, "target": "orders_clean.csv", "action": "Parsed Order Date and Ship Date as datetime", "rationale": "Consistent date type; values unchanged", "rows_affected": 0},
    {"step": 3, "target": "orders_clean.csv", "action": "Stored Postal Code as text without padding", "rationale": "Type consistency without guessing leading zeros; representation change only", "rows_affected": len(orders_clean)},
    {"step": 4, "target": "orders_clean.csv", "action": "Documented repeated Order ID + Product ID cases", "rationale": "Keep repeated order lines; no rows removed", "rows_affected": int(repeated_order_product_mask.sum())},
    {"step": 5, "target": "orders_clean.csv", "action": "Documented rows duplicated ignoring Row ID", "rationale": "Do not auto-delete duplicates; no rows removed", "rows_affected": int(duplicate_record_mask.sum())},
    {"step": 6, "target": "orders_clean.csv", "action": "Kept future-dated orders", "rationale": "Do not assume future dates are invalid", "rows_affected": int(future_order_mask.sum())},
    {"step": 7, "target": "returns_clean.csv", "action": "Left Returns at order level, unchanged", "rationale": "Avoid joins that multiply order-line records", "rows_affected": 0},
    {"step": 8, "target": "people_clean.csv", "action": "Left People unchanged", "rationale": "Region coverage validated; no change required", "rows_affected": 0},
]
change_log = pd.DataFrame(file_log_rows)
display(change_log)

change_log_path = processed_directory / "cleaning_change_log.csv"
change_log.to_csv(change_log_path, index=False)
created_files.append(change_log_path)
print("Wrote", change_log_path.relative_to(repository_root))

,step,target,action,rationale,rows_affected
0,1,orders_clean.csv,Copied all rows and columns unchanged,Preserve every record; no row deletion or corr...,0
1,2,orders_clean.csv,Parsed Order Date and Ship Date as datetime,Consistent date type; values unchanged,0
2,3,orders_clean.csv,Stored Postal Code as text without padding,Type consistency without guessing leading zero...,10194
3,4,orders_clean.csv,Documented repeated Order ID + Product ID cases,Keep repeated order lines; no rows removed,22
4,5,orders_clean.csv,Documented rows duplicated ignoring Row ID,Do not auto-delete duplicates; no rows removed,4
5,6,orders_clean.csv,Kept future-dated orders,Do not assume future dates are invalid,1106
6,7,returns_clean.csv,"Left Returns at order level, unchanged",Avoid joins that multiply order-line records,0
7,8,people_clean.csv,Left People unchanged,Region coverage validated; no change required,0


Wrote data/processed/cleaning_change_log.csv


## 11. Final Checks, Output Files, and Transactions Summary

In [10]:
raw_sha256_after = file_sha256(raw_workbook_path)
raw_unchanged = raw_sha256_after == raw_sha256_before
print("Raw workbook SHA-256 (after):", raw_sha256_after)
print("Raw workbook unchanged:", bool(raw_unchanged))
assert raw_unchanged, "Raw workbook changed during processing."

print()
print("Output files created under data/processed/:")
for output_path in created_files:
    print(f"  {output_path.relative_to(repository_root)} ({output_path.stat().st_size} bytes)")

print()
print("Transformations performed:")
print("  1. Parsed Order Date / Ship Date as datetime (values unchanged).")
print("  2. Stored Postal Code as text without padding (representation change only).")
print("  3. No rows added, removed, or corrected; no other columns changed.")
print("  4. Duplicates and repeated keys documented, not modified.")
print("  5. Returns kept at order level; People left unchanged.")

Raw workbook SHA-256 (after): 5045ed4fec49cae2cc0dfc5ab0d204ba09efa767fe00c996822499e8cc42fbbd
Raw workbook unchanged: True

Output files created under data/processed/:
  data/processed/orders_clean.csv (2472638 bytes)


  data/processed/people_clean.csv (106 bytes)
  data/processed/returns_clean.csv (5642 bytes)
  data/processed/cleaning_change_log.csv (912 bytes)

Transformations performed:
  1. Parsed Order Date / Ship Date as datetime (values unchanged).
  2. Stored Postal Code as text without padding (representation change only).
  3. No rows added, removed, or corrected; no other columns changed.
  4. Duplicates and repeated keys documented, not modified.
  5. Returns kept at order level; People left unchanged.


## 12. Notes & Limitations

- The raw workbook is never modified; processed outputs are separate copies.
- `Postal Code` is stored as text without padding. Four-character United States values are kept
  as-is; whether any originally had a leading zero cannot be determined from this dataset alone,
  and none is added.
- Repeated `Order ID` values are expected (multiple lines per order) and are preserved.
- Repeated `Order ID + Product ID` cases and rows duplicated when ignoring `Row ID` are documented
  but not removed; de-duplication would require a business decision.
- Future-dated orders are preserved; no assumption is made about their validity.
- `Product ID` and `Product Name` are not forced into a one-to-one mapping.
- `Returns` stays at order level; it is not joined onto line-level `Orders` in a way that would
  multiply order lines.
- No sales or profitability analysis is performed in this notebook.